In [1]:
import pickle
from os.path import join
import nibabel as nib
import numpy as np
import pandas as pd
from scipy import stats
from cloth_fmri.config.config import CONFIG
from cloth_fmri.utils.fmri import load_glmsingle_betas


'''

This analysis quantified model-free reliability of stimulus-evoked multivoxel response patterns. 
For each participant, response patterns were estimated separately for odd and even runs by 
averaging repeated presentations of each video within each run set. 

Reliability was then quantified by correlating odd- and even-run response patterns
across videos for each participant.

'''

'\nThis analysis quantified model-free reliability of stimulus-evoked multivoxel response patterns. \nFor each participant, response patterns were estimated separately for odd and even runs by \naveraging repeated presentations of each video within each run set. \n\nReliability was then quantified by correlating odd- and even-run response patterns\nacross videos for each participant.\n'

In [2]:
block_order_all_subs, bs_order_all_subs, scene_order_all_subs, beta_avg_all_subs = load_glmsingle_betas()


rows = []

for participant in block_order_all_subs.keys():
    for s, b, o, beta in zip(
        scene_order_all_subs[participant],
        bs_order_all_subs[participant],
        block_order_all_subs[participant],
        beta_avg_all_subs[participant],
    ):
        rows.append({
            "participant": participant,
            "scene": s,
            "bs": b,
            "order": o,
            "beta": beta,
        })

df = pd.DataFrame(rows)


df["run"] = np.tile(
    np.repeat(np.arange(1, CONFIG['runs'] + 1), CONFIG['videos_per_run']),
    len(CONFIG['subjects']),
)

In [3]:
# ----------------------------------------------------
# Compute split-half reliability
# ----------------------------------------------------
for roi_type in ['tower', 'v1', 'loc']:
    print(f"-------- ROI: {roi_type} --------")
    all_r = []

    for sub in CONFIG["subjects"]:

        # ----------------------------------------------------
        # Get ROI
        # ----------------------------------------------------

        roi_name = "sub-{}_parcel-{}.nii.gz".format(sub, roi_type)

        roi_file = join(CONFIG['roi_root'], "roi_{}/p-0.05".format(roi_type), str(sub), roi_name)


        roi_data = nib.load(roi_file)
        roi = roi_data.get_fdata()
        roi_mask = roi != 0

        # ----------------------------------------------------
        # Load beta
        # ----------------------------------------------------

        runs = CONFIG["runs"]

        dicts = CONFIG["stim_dict"]
        dicts["baseline"] = 8

        outputdir_glmsingle = join(CONFIG["glmsingle_root"], f"sub-{sub}")

        results_glmsingle = {}

        results_glmsingle["typed"] = np.load(
            join(outputdir_glmsingle, "TYPED_FITHRF_GLMDENOISE_RR.npy"),
            allow_pickle=True,
        ).item()

        betas_tmp = results_glmsingle["typed"]["betasmd"]
        data = betas_tmp

        # ----------------------------------------------------
        # Extract ROI beta
        # ----------------------------------------------------
        roi_beta = data[roi_mask, :]

        sub_mask = df["participant"] == sub
        sub_df = df.loc[sub_mask].copy()

        roi_beta_by_trial = roi_beta.T
        sub_df["beta"] = list(roi_beta_by_trial)

        # ----------------------------------------------------
        # Correlate run 1/3 vs run 2/4 per order
        # ----------------------------------------------------
        corr_rows = []
        tmp_df = sub_df

        for order, g in tmp_df.groupby("order"):
            g13 = g[g["run"].isin([1, 3])]
            g24 = g[g["run"].isin([2, 4])]

            if len(g13) == 0 or len(g24) == 0:
                continue

            beta13 = np.stack(g13["beta"].values, axis=0)
            beta24 = np.stack(g24["beta"].values, axis=0)

            avg13 = np.nanmean(beta13, axis=0)
            avg24 = np.nanmean(beta24, axis=0)

            valid = np.isfinite(avg13) & np.isfinite(avg24)

            r = np.corrcoef(avg13[valid], avg24[valid])[0, 1]

            corr_rows.append({
                "order": order,
                "scene": g["scene"].iloc[0],
                "bs": g["bs"].iloc[0],
                "n_13": len(g13),
                "n_24": len(g24),
                "corr": r,
            })

        corr_df = pd.DataFrame(corr_rows)
        mean_corr = np.mean([row["corr"] for row in corr_rows])
        all_r.append(mean_corr)


    vals = np.array(all_r)
    mean_r = np.mean(vals)
    t_stat, p_two_tailed = stats.ttest_1samp(vals, popmean=0)

    print(
        f"Average across-run split-half correlations r = {mean_r:.3f}, "
        f"One-sample t-test vs 0: "
        f"t({len(vals)-1}) = {t_stat:.2f}, "
        f"p = {p_two_tailed:.3g}"
    )

-------- ROI: tower --------
Average across-run split-half correlations r = 0.443, One-sample t-test vs 0: t(23) = 10.99, p = 1.25e-10
-------- ROI: v1 --------
Average across-run split-half correlations r = 0.580, One-sample t-test vs 0: t(23) = 14.36, p = 5.66e-13
-------- ROI: loc --------
Average across-run split-half correlations r = 0.662, One-sample t-test vs 0: t(23) = 15.24, p = 1.64e-13
